# Shadow-Tomography-Guided Hamiltonian Learning
### Derandomized Shadows, Differentiable Inversion in PennyLane, and SPAM Error Mitigation
**Author**: Jasper Sands

This tutorial demonstrates:
1. Generating randomized and derandomized classical shadow snapshots in Cirq.
2. Fast $\mathcal{O}(k)$ Pauli observable estimation without $2^N \times 2^N$ matrix overhead.
3. Inverting snapshot statistics through thermal Gibbs states to learn unknown coupling parameters $J_{ij}$.
4. Applying SPAM readout error mitigation via inverted confusion matrix convolution.
5. Streaming continuous parameter tracking with an Extended Kalman Filter (EKF).

In [ ]:
import numpy as np
import cirq
from shadow_learning import (
    measure_random_pauli_shadows,
    estimate_many_observables,
    learn_hamiltonian_from_shadows,
    DerandomizedShadowSelector,
)

# 1. Collect Classical Shadows on 2-Qubit Bell State
q = cirq.LineQubit.range(2)
c = cirq.Circuit([cirq.H(q[0]), cirq.CNOT(q[0], q[1])])
snapshots = measure_random_pauli_shadows(c, n_qubits=2, n_snapshots=1000, seed=42)

# 2. Reconstruct Pauli Observables
obs = estimate_many_observables(snapshots, ["ZZ", "XX", "YY", "ZI", "IZ"])
print("Reconstructed Observables:")
for k, v in obs.items():
    print(f"  <{k}> = {v:.4f}")

In [ ]:
# 3. Differentiable Hamiltonian Parameter Inversion
res = learn_hamiltonian_from_shadows(snapshots, n_qubits=2, beta=1.0)
print(f"Recovered Coupling J_01: {res.recovered_j_matrix[0, 1]:.4f}")
print(f"Recovered Field h: {res.recovered_h_vector}")
print(f"Inversion Loss: {res.final_loss:.6e}")